In [59]:
import ast
from collections import Counter
import heapq
from itertools import chain
import json
from pathlib import Path
import random

from deepctr_torch.inputs import DenseFeat, SparseFeat, VarLenSparseFeat, get_feature_names
from deepctr_torch.models import DeepFM
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, diags
from sklearn.preprocessing import LabelEncoder, StandardScaler
import torch

Ocupe ia para la evaluacion de metricas y generar resultados apra la competencia

In [60]:
dataset_path = Path('../dataset')

# Cargar interacciones separadas y metadatos
train_df = pd.read_csv(dataset_path / 'train.csv')
val_df = pd.read_csv(dataset_path / 'validation.csv')
item_df = pd.read_csv(dataset_path / 'item_metadata.csv')

print(f"Train: {len(train_df)} filas | Val: {len(val_df)} filas | Catálogo: {len(item_df)} recetas")

Train: 157108 filas | Val: 27416 filas | Catálogo: 7047 recetas


In [61]:
# 1. Parseo seguro de listas
def parse_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, str):
        try:
            val = ast.literal_eval(x)
            return val if isinstance(val, list) else []
        except (ValueError, SyntaxError):
            return []
    return []

def parse_nutrition(x):
    parsed = parse_list(x)
    return parsed if len(parsed) == 7 else [0.0] * 7

item_df['ingredients'] = item_df['ingredients'].apply(parse_list)
item_df['tags'] = item_df['tags'].apply(parse_list)
item_df['nutrition_parsed'] = item_df['nutrition'].apply(parse_nutrition)

# Expandir nutrientes
nutri_cols = ['calories', 'total_fat', 'sugar', 'sodium', 'protein', 'sat_fat', 'carbs']
item_df[nutri_cols] = pd.DataFrame(item_df['nutrition_parsed'].tolist(), index=item_df.index)

# 2. Vocabularios (min_df=5 para ingredientes)
ing_counts = Counter(chain.from_iterable(item_df['ingredients']))
valid_ings = {ing for ing, count in ing_counts.items() if count >= 5}

ing_to_id = {ing: i + 1 for i, ing in enumerate(valid_ings)}
tag_to_id = {tag: i + 1 for i, tag in enumerate(Counter(chain.from_iterable(item_df['tags'])))}

MAX_ING = 15
MAX_TAGS = 10

def encode_and_pad(seq, vocab_map, max_len):
    encoded = [vocab_map[item] for item in seq if item in vocab_map]
    if len(encoded) > max_len:
        return encoded[:max_len]
    return encoded + [0] * (max_len - len(encoded))

item_df['ingredients_seq'] = item_df['ingredients'].apply(lambda x: encode_and_pad(x, ing_to_id, MAX_ING))
item_df['tags_seq'] = item_df['tags'].apply(lambda x: encode_and_pad(x, tag_to_id, MAX_TAGS))

In [62]:
# Mapeo de IDs (construido con train para users y catálogo para recetas)
user_map = {uid: idx for idx, uid in enumerate(train_df['user_id'].unique())}
recipe_map = {rid: idx for idx, rid in enumerate(item_df['id'].values)}

dense_features = ['minutes', 'n_steps'] + nutri_cols

# Ajustar scaler únicamente sobre item_metadata
item_df_norm = item_df.copy()
item_df_norm['minutes'] = np.log1p(item_df_norm['minutes'].clip(upper=1440))
scaler = StandardScaler()
scaler.fit(item_df_norm[dense_features].to_numpy())

# Definición de columnas de DeepCTR (Todos los embeddings = 32)
EMBED_DIM = 32

sparse_feature_columns = [
    SparseFeat('user_id', vocabulary_size=len(user_map) + 1, embedding_dim=EMBED_DIM),
    SparseFeat('recipe_id', vocabulary_size=len(recipe_map) + 1, embedding_dim=EMBED_DIM)
]

dense_feature_columns = [DenseFeat(feat, 1) for feat in dense_features]

varlen_feature_columns = [
    VarLenSparseFeat(
        SparseFeat('ingredients_seq', vocabulary_size=len(ing_to_id) + 1, embedding_dim=EMBED_DIM),
        maxlen=MAX_ING,
        combiner='sum'
    ),
    VarLenSparseFeat(
        SparseFeat('tags_seq', vocabulary_size=len(tag_to_id) + 1, embedding_dim=EMBED_DIM),
        maxlen=MAX_TAGS,
        combiner='sum'
    )
]

linear_feature_columns = sparse_feature_columns + dense_feature_columns + varlen_feature_columns
dnn_feature_columns = sparse_feature_columns + dense_feature_columns + varlen_feature_columns
feature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)

In [63]:
recipe_meta = item_df.set_index('id').to_dict(orient='index')

def build_model_input(interaction_df: pd.DataFrame):
    n_samples = len(interaction_df)
    b_uids, b_rids, b_ing, b_tag, b_dense = [], [], [], [], []

    default_ing = [0] * MAX_ING
    default_tag = [0] * MAX_TAGS
    default_dense = [0.0] * len(dense_features)

    UNK_USER = len(user_map)
    UNK_RECIPE = len(recipe_map)

    for row in interaction_df.itertuples(index=False):
        uid, rid = row.user_id, row.recipe_id
        b_uids.append(user_map.get(uid, UNK_USER))
        b_rids.append(recipe_map.get(rid, UNK_RECIPE))

        meta = recipe_meta.get(rid)
        if meta is not None:
            b_ing.append(meta.get('ingredients_seq', default_ing))
            b_tag.append(meta.get('tags_seq', default_tag))
            b_dense.append([meta.get(col, 0.0) for col in dense_features])
        else:
            b_ing.append(default_ing)
            b_tag.append(default_tag)
            b_dense.append(default_dense)

    model_input = {
        'user_id': np.array(b_uids, dtype=np.int32),
        'recipe_id': np.array(b_rids, dtype=np.int32),
        'ingredients_seq': np.vstack(b_ing),
        'tags_seq': np.vstack(b_tag),
    }

    # Transformar columnas densas
    dense_arr = np.array(b_dense, dtype=np.float32)
    dense_arr[:, 0] = np.log1p(np.clip(dense_arr[:, 0], 0, 1440))
    scaled_dense = scaler.transform(dense_arr)

    for i, col in enumerate(dense_features):
        model_input[col] = scaled_dense[:, i]

    ratings = interaction_df['rating'].values.astype(np.float32)
    return model_input, ratings

train_model_input, train_ratings = build_model_input(train_df)
val_model_input, val_ratings = build_model_input(val_df)

In [64]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

model = DeepFM(
    linear_feature_columns,
    dnn_feature_columns,
    task='regression',
    device=device
)

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mse', 'mae']
)

history = model.fit(
    train_model_input,
    train_ratings,
    batch_size=4096,
    epochs=10,
    validation_data=(val_model_input, val_ratings),
    verbose=2
)

cpu
Train on 157108 samples, validate on 27416 samples, 39 steps per epoch
Epoch 1/10
2s - loss:  9.7564 - mse:  9.6157 - val_mse:  1.4623
Epoch 2/10
2s - loss:  0.9470 - mse:  0.9448 - val_mse:  0.8599
Epoch 3/10
2s - loss:  0.6266 - mse:  0.6267 - val_mse:  0.7981
Epoch 4/10
2s - loss:  0.5820 - mse:  0.5815 - val_mse:  0.7927
Epoch 5/10
2s - loss:  0.5718 - mse:  0.5726 - val_mse:  0.7957
Epoch 6/10
2s - loss:  0.5693 - mse:  0.5694 - val_mse:  0.7972
Epoch 7/10
2s - loss:  0.5681 - mse:  0.5684 - val_mse:  0.7986
Epoch 8/10
2s - loss:  0.5672 - mse:  0.5675 - val_mse:  0.7987
Epoch 9/10
2s - loss:  0.5673 - mse:  0.5683 - val_mse:  0.7999
Epoch 10/10
2s - loss:  0.5670 - mse:  0.5674 - val_mse:  0.8011


In [ ]:
def evaluate_deepfm_metrics(
    *,
    deepfm_model,
    train_df: pd.DataFrame,
    item_df: pd.DataFrame,
    val_df: pd.DataFrame,
    scaler,
    dense_features: list[str],
    max_ing: int = 15,
    max_tags: int = 10,
    k: int = 10,
    threshold: float = 4.0,
    batch_size: int = 8192,
) -> dict[str, float]:
    user_map = {uid: idx for idx, uid in enumerate(train_df['user_id'].unique())}
    recipe_map = {rid: idx for idx, rid in enumerate(item_df['id'].values)}
    recipe_meta = item_df.set_index('id').to_dict(orient='index')

    default_ing = [0] * max_ing
    default_tag = [0] * max_tags
    default_dense = [0.0] * len(dense_features)

    # 1. Construir testset_rating y testset_ranking
    testset_rating = list(val_df[['user_id', 'recipe_id', 'rating']].itertuples(index=False, name=None))

    rng = random.Random(42)
    all_items = set(train_df['recipe_id'].unique())
    user_seen_train = train_df.groupby('user_id')['recipe_id'].apply(set).to_dict()
    user_seen_val = val_df.groupby('user_id')['recipe_id'].apply(set).to_dict()

    testset_ranking = []
    for row in val_df.itertuples(index=False):
        testset_ranking.append((row.user_id, row.recipe_id, float(row.rating)))
    for uid, val_items in user_seen_val.items():
        seen = user_seen_train.get(uid, set()) | val_items
        unseen = list(all_items - seen)
        for iid in rng.sample(unseen, min(len(unseen), 100)):
            testset_ranking.append((uid, iid, 0.0))

    def predict_batch(samples: list[tuple]) -> np.ndarray:
        n_samples = len(samples)
        preds_all = np.empty(n_samples, dtype=np.float32)

        for start in range(0, n_samples, batch_size):
            end = min(start + batch_size, n_samples)
            chunk = samples[start:end]

            b_uids, b_rids, b_ing, b_tag, b_dense = [], [], [], [], []
            for uid, iid, _ in chunk:
                b_uids.append(user_map.get(uid, 0))
                b_rids.append(recipe_map.get(iid, 0))

                meta = recipe_meta.get(iid)
                if meta is not None:
                    b_ing.append(meta.get('ingredients_seq', default_ing))
                    b_tag.append(meta.get('tags_seq', default_tag))
                    b_dense.append([meta.get(col, 0.0) for col in dense_features])
                else:
                    b_ing.append(default_ing)
                    b_tag.append(default_tag)
                    b_dense.append(default_dense)

            batch_dict = {
                'user_id': np.array(b_uids, dtype=np.int32),
                'recipe_id': np.array(b_rids, dtype=np.int32),
                'ingredients_seq': np.vstack(b_ing),
                'tags_seq': np.vstack(b_tag),
            }

            dense_arr = np.array(b_dense, dtype=np.float32)
            dense_arr[:, 0] = np.log1p(np.clip(dense_arr[:, 0], 0, 1440))
            scaled = scaler.transform(dense_arr)
            for i, col in enumerate(dense_features):
                batch_dict[col] = scaled[:, i]

            preds = deepfm_model.predict(batch_dict, batch_size=len(chunk))
            preds_all[start:end] = np.clip(preds.squeeze(), 0.0, 5.0)

        return preds_all

    est_ratings = predict_batch(testset_rating)
    true_ratings = np.fromiter((r for _, _, r in testset_rating), dtype=np.float32)
    err = est_ratings - true_ratings
    rmse_val = float(np.sqrt(np.mean(err**2)))
    mae_val = float(np.mean(np.abs(err)))

    # 3. Ranking Metrics
    est_ranking = predict_batch(testset_ranking)
    uids = [u for u, _, _ in testset_ranking]
    iids = [i for _, i, _ in testset_ranking]
    true_ranking = np.fromiter((r for _, _, r in testset_ranking), dtype=np.float32)

    df_preds = pd.DataFrame({'uid': uids, 'iid': iids, 'est_r': est_ranking, 'true_r': true_ranking})
    df_preds['is_rel'] = (df_preds['true_r'] >= threshold).astype(np.int8)

    df_sorted = df_preds.sort_values(by=['uid', 'est_r'], ascending=[True, False])
    df_sorted['rank'] = df_sorted.groupby('uid').cumcount() + 1
    top_k_df = df_sorted[df_sorted['rank'] <= k].copy()

    discounts = 1.0 / np.log2(top_k_df['rank'].to_numpy() + 1)
    top_k_df['dcg_contrib'] = top_k_df['is_rel'].to_numpy() * discounts
    top_k_df['cum_rel'] = top_k_df.groupby('uid')['is_rel'].cumsum()
    top_k_df['prec_at_r'] = top_k_df['cum_rel'] / top_k_df['rank']
    top_k_df['ap_contrib'] = top_k_df['prec_at_r'] * top_k_df['is_rel']

    user_metrics = top_k_df.groupby('uid').agg(
        hits=('is_rel', 'sum'),
        dcg=('dcg_contrib', 'sum'),
        ap_sum=('ap_contrib', 'sum'),
    )
    user_totals = df_preds.groupby('uid')['is_rel'].sum().rename('total_rel')
    valid_metrics = user_metrics.join(user_totals, how='right').fillna(0)
    valid_metrics = valid_metrics[valid_metrics['total_rel'] > 0].copy()

    if len(valid_metrics) > 0:
        valid_metrics['precision'] = valid_metrics['hits'] / k
        valid_metrics['recall'] = valid_metrics['hits'] / valid_metrics['total_rel']
        ideal_hits = np.minimum(valid_metrics['total_rel'].to_numpy(dtype=int), k)
        ideal_discounts = 1.0 / np.log2(np.arange(2, k + 2))
        cum_idcg_pad = np.insert(np.cumsum(ideal_discounts), 0, 0.0)
        valid_metrics['idcg'] = cum_idcg_pad[ideal_hits]
        valid_metrics['ndcg'] = np.where(
            valid_metrics['idcg'] > 0, valid_metrics['dcg'] / valid_metrics['idcg'], 0.0
        )
        valid_metrics['map'] = valid_metrics['ap_sum'] / ideal_hits

        mean_precision = float(valid_metrics['precision'].mean())
        mean_recall = float(valid_metrics['recall'].mean())
        mean_ndcg = float(valid_metrics['ndcg'].mean())
        mean_map = float(valid_metrics['map'].mean())
    else:
        mean_precision = mean_recall = mean_ndcg = mean_map = 0.0

    item_counts = train_df['recipe_id'].value_counts().to_dict()
    top_k_df['train_count'] = top_k_df['iid'].map(item_counts).fillna(0.1)
    probs = top_k_df['train_count'].to_numpy() / len(train_df)
    novelty_val = float(np.mean(-np.log2(probs))) if len(top_k_df) > 0 else 0.0

    raw_to_inner = {rid: i for i, rid in enumerate(train_df['recipe_id'].unique())}
    u_to_inner = {uid: i for i, uid in enumerate(train_df['user_id'].unique())}
    rows = train_df['recipe_id'].map(raw_to_inner).values
    cols = train_df['user_id'].map(u_to_inner).values

    train_sparse = csr_matrix(
        (np.ones(len(rows), dtype=np.float32), (rows, cols)),
        shape=(len(raw_to_inner), len(u_to_inner)),
    )
    row_sums = np.asarray(train_sparse.power(2).sum(axis=1)).flatten()
    inv_norms = np.zeros_like(row_sums)
    nonzero = row_sums > 0
    inv_norms[nonzero] = 1.0 / np.sqrt(row_sums[nonzero])
    train_norm = diags(inv_norms).dot(train_sparse).tocsr()

    user_rec_items = top_k_df.groupby('uid')['iid'].apply(
        lambda items: [raw_to_inner[iid] for iid in items if iid in raw_to_inner]
    )

    diversities = []
    for inner_list in user_rec_items:
        n_rec = len(inner_list)
        if n_rec <= 1:
            diversities.append(1.0)
            continue
        sub_mat = train_norm[inner_list, :]
        sim_mat = (sub_mat @ sub_mat.T).toarray()
        sum_sim = sim_mat.sum() - np.trace(sim_mat)
        diversities.append(1.0 - float(sum_sim / (n_rec * (n_rec - 1))))

    diversity_val = float(np.mean(diversities)) if diversities else 1.0

    return {
        'RMSE': round(rmse_val, 4),
        'MAE': round(mae_val, 4),
        f'Recall@{k}': round(mean_recall, 4),
        f'Precision@{k}': round(mean_precision, 4),
        f'NDCG@{k}': round(mean_ndcg, 4),
        f'MAP@{k}': round(mean_map, 4),
        'Diversity': round(diversity_val, 4),
        'Novelty': round(novelty_val, 4),
    }

metrics_deepfm = evaluate_deepfm_metrics(
    deepfm_model=model,
    train_df=train_df,
    item_df=item_df,
    val_df=val_df,
    scaler=scaler,
    dense_features=dense_features,
    k=10,
    threshold=4.0
)

pd.DataFrame([metrics_deepfm], index=['DeepFM'])

,RMSE,MAE,Recall@10,Precision@10,NDCG@10,MAP@10,Diversity,Novelty
DeepFM,0.89,0.4747,0.2288,0.1252,0.2748,0.2163,0.9856,13.129


In [ ]:
test_fill_df = pd.read_csv(dataset_path / 'rating_template_fill.csv')

# Formatear el input para DeepFM
n_fill = len(test_fill_df)
b_uids, b_rids, b_ing, b_tag, b_dense = [], [], [], [], []

default_ing = [0] * MAX_ING
default_tag = [0] * MAX_TAGS
default_dense = [0.0] * len(dense_features)

for row in test_fill_df.itertuples(index=False):
    b_uids.append(user_map.get(row.user_id, 0))
    b_rids.append(recipe_map.get(row.recipe_id, 0))

    meta = recipe_meta.get(row.recipe_id)
    if meta is not None:
        b_ing.append(meta.get('ingredients_seq', default_ing))
        b_tag.append(meta.get('tags_seq', default_tag))
        b_dense.append([meta.get(col, 0.0) for col in dense_features])
    else:
        b_ing.append(default_ing)
        b_tag.append(default_tag)
        b_dense.append(default_dense)

fill_input = {
    'user_id': np.array(b_uids, dtype=np.int32),
    'recipe_id': np.array(b_rids, dtype=np.int32),
    'ingredients_seq': np.vstack(b_ing),
    'tags_seq': np.vstack(b_tag),
}

dense_arr = np.array(b_dense, dtype=np.float32)
dense_arr[:, 0] = np.log1p(np.clip(dense_arr[:, 0], 0, 1440))
scaled = scaler.transform(dense_arr)
for i, col in enumerate(dense_features):
    fill_input[col] = scaled[:, i]

raw_preds = model_full.predict(fill_input, batch_size=4096)
test_fill_df['rating'] = np.clip(raw_preds.squeeze(), 1.0, 5.0)
test_fill_df.to_csv(dataset_path / 'competicion_rating_completado.csv', index=False)
print("Archivo 'competicion_rating_completado.csv' generado exitosamente.")

Archivo 'competicion_rating_completado.csv' generado exitosamente.


In [ ]:
with open(dataset_path / 'ranking_template_fill.json', encoding='utf-8') as f:
    target_users_dict = json.load(f)

all_catalog_items = [int(x) for x in train_df['recipe_id'].unique()]
user_seen_train = train_df.groupby('user_id')['recipe_id'].apply(lambda s: set(int(x) for x in s)).to_dict()

top_k_recommendations = {}

for str_uid in target_users_dict.keys():
    uid = int(str_uid) if str_uid.isdigit() else str_uid
    seen = user_seen_train.get(uid, set())
    candidates = [iid for iid in all_catalog_items if iid not in seen]

    # Predecir candidates en bloque
    c_uids = [user_map.get(uid, 0)] * len(candidates)
    c_rids = [recipe_map.get(iid, 0) for iid in candidates]
    c_ing = [recipe_meta.get(iid, {}).get('ingredients_seq', default_ing) for iid in candidates]
    c_tag = [recipe_meta.get(iid, {}).get('tags_seq', default_tag) for iid in candidates]
    c_dense = [[recipe_meta.get(iid, {}).get(col, 0.0) for col in dense_features] for iid in candidates]

    cand_input = {
        'user_id': np.array(c_uids, dtype=np.int32),
        'recipe_id': np.array(c_rids, dtype=np.int32),
        'ingredients_seq': np.vstack(c_ing),
        'tags_seq': np.vstack(c_tag),
    }

    dense_arr = np.array(c_dense, dtype=np.float32)
    dense_arr[:, 0] = np.log1p(np.clip(dense_arr[:, 0], 0, 1440))
    scaled = scaler.transform(dense_arr)
    for i, col in enumerate(dense_features):
        cand_input[col] = scaled[:, i]

    scores = model.predict(cand_input, batch_size=4096).squeeze()
    user_preds = list(zip(candidates, scores))

    top_10 = [iid for iid, _ in heapq.nlargest(10, user_preds, key=lambda x: x[1])]
    top_k_recommendations[str(str_uid)] = top_10

with open(dataset_path / 'competicion_ranking.json', 'w', encoding='utf-8') as f:
    json.dump(top_k_recommendations, f, indent=4)

print("Archivo 'competicion_ranking.json' generado exitosamente.")

Archivo 'competicion_ranking.json' generado exitosamente.
